# Data Consistency and Standardization

## Objective

In this notebook, we will cover:

### Text Cleaning
- Convert to Lowercase
- Remove Extra Spaces
- Remove Special Characters
- Unicode Normalization
- Spelling Correction

### Numerical Scaling
- Min-Max Scaling
- Standard Scaling
- Robust Scaling
- Log Transformation
- Power Transformation

The original dataset will remain unchanged.

In [4]:
import numpy as np
import pandas as pd
import unicodedata

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    PowerTransformer
)

In [5]:
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [6]:
# Create a copy for practice
df_clean = df.copy()

print("Dataset Shape:", df_clean.shape)
print("Original Dataset Preserved:", df.equals(df_clean))

Dataset Shape: (299, 10)
Original Dataset Preserved: True


# Text Cleaning

The Heart Failure dataset does not contain suitable text columns for demonstrating all text-cleaning techniques.

Therefore, a small practice text column will be created without modifying the original dataset.

In [7]:
practice_text = pd.DataFrame({
    "city": [
        " LAHORE ",
        "lahore",
        "Lahor!",
        " KARACHI",
        "karachi#",
        "Islamabad ",
        "ISLAMABAD"
    ]
})

practice_text

,city
0,LAHORE
1,lahore
2,Lahor!
3,KARACHI
4,karachi#
5,Islamabad
6,ISLAMABAD


In [8]:
practice_text["city_lower"] = (
    practice_text["city"]
    .str.lower()
)

practice_text[
    ["city", "city_lower"]
]

,city,city_lower
0,LAHORE,lahore
1,lahore,lahore
2,Lahor!,lahor!
3,KARACHI,karachi
4,karachi#,karachi#
5,Islamabad,islamabad
6,ISLAMABAD,islamabad


In [9]:
practice_text["city_no_spaces"] = (
    practice_text["city_lower"]
    .str.strip()
)

practice_text[
    ["city_lower", "city_no_spaces"]
]

,city_lower,city_no_spaces
0,lahore,lahore
1,lahore,lahore
2,lahor!,lahor!
3,karachi,karachi
4,karachi#,karachi#
5,islamabad,islamabad
6,islamabad,islamabad


In [10]:
practice_text["city_clean"] = (
    practice_text["city_no_spaces"]
    .str.replace(
        r"[^a-zA-Z0-9\s]",
        "",
        regex=True
    )
)

practice_text[
    ["city_no_spaces", "city_clean"]
]

,city_no_spaces,city_clean
0,lahore,lahore
1,lahore,lahore
2,lahor!,lahor
3,karachi,karachi
4,karachi#,karachi
5,islamabad,islamabad
6,islamabad,islamabad


## Unicode Normalization

Unicode Normalization converts different Unicode representations into a consistent standard form.

In [11]:
practice_text["city_unicode"] = (
    practice_text["city_clean"]
    .apply(
        lambda text: unicodedata.normalize(
            "NFKC",
            text
        )
    )
)

practice_text[
    ["city_clean", "city_unicode"]
]

,city_clean,city_unicode
0,lahore,lahore
1,lahore,lahore
2,lahor,lahor
3,karachi,karachi
4,karachi,karachi
5,islamabad,islamabad
6,islamabad,islamabad


## Spelling Correction

Known spelling inconsistencies can be corrected using a validated mapping.

Automatic spelling correction should not be applied blindly because similar words may represent different valid categories.

In [12]:
spelling_corrections = {
    "lahor": "lahore"
}

practice_text["city_final"] = (
    practice_text["city_unicode"]
    .replace(spelling_corrections)
)

practice_text[
    ["city", "city_final"]
]

,city,city_final
0,LAHORE,lahore
1,lahore,lahore
2,Lahor!,lahore
3,KARACHI,karachi
4,karachi#,karachi
5,Islamabad,islamabad
6,ISLAMABAD,islamabad


In [13]:
print("Original Categories:")
print(practice_text["city"].unique())

print("\nCleaned Categories:")
print(practice_text["city_final"].unique())

Original Categories:
<StringArray>
[  ' LAHORE ',     'lahore',     'Lahor!',   ' KARACHI',   'karachi#',
 'Islamabad ',  'ISLAMABAD']
Length: 7, dtype: str

Cleaned Categories:
<StringArray>
['lahore', 'karachi', 'islamabad']
Length: 3, dtype: str


# Numerical Scaling

Numerical features can have very different measurement scales.

For this dataset, we will demonstrate scaling using continuous numerical features.

Binary categorical columns such as `diabetes`, `sex`, `anaemia`, and `high_blood_pressure` will not be included.

In [14]:
numerical_features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]

df[numerical_features].describe()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium
count,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000
mean,60.833893,581.839465,38.083612,263358.029264,1.39388,136.625418
std,11.894809,970.287881,11.834841,97804.236869,1.03451,4.412477
min,40.000000,23.000000,14.000000,25100.000000,0.50000,113.000000
25%,51.000000,116.500000,30.000000,212500.000000,0.90000,134.000000
50%,60.000000,250.000000,38.000000,262000.000000,1.10000,137.000000
75%,70.000000,582.000000,45.000000,303500.000000,1.40000,140.000000
max,95.000000,7861.000000,80.000000,850000.000000,9.40000,148.000000


## Min-Max Scaling

Min-Max Scaling transforms numerical values to a fixed range, usually between 0 and 1.

In [15]:
minmax_scaler = MinMaxScaler()

minmax_values = minmax_scaler.fit_transform(
    df[numerical_features]
)

df_minmax = pd.DataFrame(
    minmax_values,
    columns=numerical_features
)

df_minmax.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium
0,0.636364,0.071319,0.090909,0.290823,0.157303,0.485714
1,0.272727,1.000000,0.363636,0.288833,0.067416,0.657143
2,0.454545,0.015693,0.090909,0.165960,0.089888,0.457143
3,0.181818,0.011227,0.090909,0.224148,0.157303,0.685714
4,0.454545,0.017479,0.090909,0.365984,0.247191,0.085714


In [16]:
print("Minimum Values:")
print(df_minmax.min())

print("\nMaximum Values:")
print(df_minmax.max())

Minimum Values:
age                         0.0
creatinine_phosphokinase    0.0
ejection_fraction           0.0
platelets                   0.0
serum_creatinine            0.0
serum_sodium                0.0
dtype: float64

Maximum Values:
age                         1.0
creatinine_phosphokinase    1.0
ejection_fraction           1.0
platelets                   1.0
serum_creatinine            1.0
serum_sodium                1.0
dtype: float64


## Standard Scaling

Standard Scaling centers features around zero using their mean and standard deviation.

In [17]:
standard_scaler = StandardScaler()

standard_values = standard_scaler.fit_transform(
    df[numerical_features]
)

df_standard = pd.DataFrame(
    standard_values,
    columns=numerical_features
)

df_standard.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium
0,1.192945,0.000166,-1.530560,1.681648e-02,0.490057,-1.504036
1,-0.491279,7.514640,-0.007077,7.535660e-09,-0.284552,-0.141976
2,0.350833,-0.449939,-1.530560,-1.038073e+00,-0.090900,-1.731046
3,-0.912335,-0.486071,-1.530560,-5.464741e-01,0.490057,0.085034
4,0.350833,-0.435486,-1.530560,6.517986e-01,1.264666,-4.682176


In [18]:
print("Means:")
print(
    df_standard.mean().round(2)
)

print("\nStandard Deviations:")
print(
    df_standard.std(ddof=0).round(2)
)

Means:
age                         0.0
creatinine_phosphokinase    0.0
ejection_fraction          -0.0
platelets                   0.0
serum_creatinine            0.0
serum_sodium               -0.0
dtype: float64

Standard Deviations:
age                         1.0
creatinine_phosphokinase    1.0
ejection_fraction           1.0
platelets                   1.0
serum_creatinine            1.0
serum_sodium                1.0
dtype: float64


## Robust Scaling

Robust Scaling uses the median and IQR, making it less sensitive to extreme values.

In [19]:
robust_scaler = RobustScaler()

robust_values = robust_scaler.fit_transform(
    df[numerical_features]
)

df_robust = pd.DataFrame(
    robust_values,
    columns=numerical_features
)

df_robust.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium
0,0.789474,0.713212,-1.2,0.032967,1.6,-1.166667
1,-0.263158,16.350161,0.0,0.014923,0.0,-0.166667
2,0.263158,-0.223416,-1.2,-1.098901,0.4,-1.333333
3,-0.526316,-0.298604,-1.2,-0.571429,1.6,0.000000
4,0.263158,-0.193340,-1.2,0.714286,3.2,-3.500000


In [20]:
scaling_comparison = pd.DataFrame({
    "Original": df["serum_creatinine"],
    "MinMax": df_minmax["serum_creatinine"],
    "Standard": df_standard["serum_creatinine"],
    "Robust": df_robust["serum_creatinine"]
})

scaling_comparison.head(10)

,Original,MinMax,Standard,Robust
0,1.9,0.157303,0.490057,1.6
1,1.1,0.067416,-0.284552,0.0
2,1.3,0.089888,-0.090900,0.4
3,1.9,0.157303,0.490057,1.6
4,2.7,0.247191,1.264666,3.2
5,2.1,0.179775,0.683709,2.0
6,1.2,0.078652,-0.187726,0.2
7,1.1,0.067416,-0.284552,0.0
8,1.5,0.112360,0.102752,0.8
9,9.4,1.000000,7.752020,16.6


## Log Transformation

Log Transformation can reduce right-skewness and compress large positive values.

It is not the same as ordinary feature scaling.

In [21]:
log_feature = "creatinine_phosphokinase"

df_log = df.copy()

df_log[
    "creatinine_phosphokinase_log"
] = np.log1p(
    df_log[log_feature]
)

df_log[
    [
        log_feature,
        "creatinine_phosphokinase_log"
    ]
].head(10)

,creatinine_phosphokinase,creatinine_phosphokinase_log
0,582,6.368187
1,7861,8.969796
2,146,4.990433
3,111,4.718499
4,160,5.081404
5,47,3.871201
6,246,5.509388
7,315,5.755742
8,157,5.062595
9,123,4.820282


In [22]:
original_skew = df[
    log_feature
].skew()

log_skew = df_log[
    "creatinine_phosphokinase_log"
].skew()

print(
    "Original Skewness:",
    round(original_skew, 2)
)

print(
    "After Log Transformation:",
    round(log_skew, 2)
)

Original Skewness: 4.46
After Log Transformation: 0.42


## Power Transformation

Power Transformation attempts to make numerical distributions more symmetric.

We will use the Yeo-Johnson method.

In [23]:
power_feature = "creatinine_phosphokinase"

power_transformer = PowerTransformer(
    method="yeo-johnson"
)

power_values = power_transformer.fit_transform(
    df[[power_feature]]
)

df_power = df.copy()

df_power[
    "creatinine_phosphokinase_power"
] = power_values.flatten()

df_power[
    [
        power_feature,
        "creatinine_phosphokinase_power"
    ]
].head()

,creatinine_phosphokinase,creatinine_phosphokinase_power
0,582,0.691615
1,7861,2.401701
2,146,-0.553424
3,111,-0.833885
4,160,-0.462335


In [24]:
power_skew = df_power[
    "creatinine_phosphokinase_power"
].skew()

print(
    "Original Skewness:",
    round(original_skew, 2)
)

print(
    "Log Transformation:",
    round(log_skew, 2)
)

print(
    "Power Transformation:",
    round(power_skew, 2)
)

Original Skewness: 4.46
Log Transformation: 0.42
Power Transformation: 0.04


In [25]:
final_comparison = pd.DataFrame({
    "Original": df["creatinine_phosphokinase"],
    "MinMax": df_minmax["creatinine_phosphokinase"],
    "Standard": df_standard["creatinine_phosphokinase"],
    "Robust": df_robust["creatinine_phosphokinase"],
    "Log": df_log["creatinine_phosphokinase_log"],
    "Power": df_power["creatinine_phosphokinase_power"]
})

final_comparison.head(10)

,Original,MinMax,Standard,Robust,Log,Power
0,582,0.071319,0.000166,0.713212,6.368187,0.691615
1,7861,1.000000,7.514640,16.350161,8.969796,2.401701
2,146,0.015693,-0.449939,-0.223416,4.990433,-0.553424
3,111,0.011227,-0.486071,-0.298604,4.718499,-0.833885
4,160,0.017479,-0.435486,-0.193340,5.081404,-0.462335
5,47,0.003062,-0.552141,-0.436090,3.871201,-1.791715
6,246,0.028451,-0.346704,-0.008593,5.509388,-0.051432
7,315,0.037254,-0.275472,0.139635,5.755742,0.172437
8,157,0.017096,-0.438583,-0.199785,5.062595,-0.481058
9,123,0.012758,-0.473683,-0.272825,4.820282,-0.727455


In [26]:
print("Original Shape:", df.shape)

print(
    "Original Dataset Missing Values:",
    df.isnull().sum().sum()
)

print(
    "Original Columns:",
    df.columns.tolist()
)

Original Shape: (299, 10)
Original Dataset Missing Values: 0
Original Columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex']


# Summary

In this notebook, we covered:

## Text Cleaning

- Lowercase Conversion
- Extra Space Removal
- Special Character Removal
- Unicode Normalization
- Spelling Correction

## Numerical Scaling and Transformation

- Min-Max Scaling
- Standard Scaling
- Robust Scaling
- Log Transformation
- Power Transformation

## Key Learnings

- Text values should follow consistent formatting.
- Binary categorical features should not be treated as continuous measurements.
- Min-Max Scaling usually maps values between 0 and 1.
- Standard Scaling centers features around zero.
- Robust Scaling is less sensitive to outliers.
- Log Transformation can reduce right-skewness.
- Power Transformation can make distributions more symmetric.
- The original dataset should be preserved during experimentation.
- In a real ML pipeline, scalers should be fitted on training data only.